In [37]:
import os
from pyspark import SparkContext,SparkConf
from pyspark.sql import SparkSession
try:
    sc.stop()
except:
    pass

In [38]:
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

conf = (SparkConf()
        .setMaster("local[*]")        # Corre Spark en tu máquina local, usando todos los núcleos disponibles del CPU ("local" a secas, corre Spark en modo local con un solo hilo (1 CPU))
        .setAppName("2.3.1 MapReduce con Spark")
        .set("spark.driver.bindAddress", "127.0.0.1")
        .set("spark.driver.host", "127.0.0.1"))
sc = SparkContext(conf = conf)
sc


<SparkContext master=local[*] appName=2.3.1 MapReduce con Spark>

In [39]:
spark = SparkSession.builder \
    .appName("Spark") \
    .getOrCreate()


### Procedimiento 1: ¿Cuál es el sueldo promedio de los médicos según su especialidad?

In [40]:
# Cargamos los datos de médico:

df_medicos = spark.read.csv("./datos/medico",header=True)

# Los convertimos a dataset distribuido:

rdd_medicos = df_medicos.rdd 

In [41]:
df_medicos.show()

+-----------+----------+---------------+--------------------+
|       cuil|   salario|cuil_supervisor|        especialidad|
+-----------+----------+---------------+--------------------+
|30300689962|1782910.14|           NULL|        Neumonologia|
|20427108705|3296132.08|           NULL|      Endocrinologia|
|30342958162| 940146.66|           NULL|   Gastroenterologia|
|27151521354| 958256.44|           NULL|      Endocrinologia|
|27174348300|2940407.64|           NULL|        Neumonologia|
|27153690187|3037886.98|           NULL|           Pediatria|
|20145084661|2617656.63|           NULL|Otorrinolaringologia|
|23346067198|2841246.24|           NULL|           Pediatria|
|20263139686|3274857.42|           NULL|         Ginecologia|
|27146393286|1860174.85|           NULL|   Gastroenterologia|
|23199216046|1257982.26|           NULL|         Psiquiatria|
|20240570059|2174826.10|           NULL|          Neurologia|
|20332040918|2964808.55|           NULL|      Clinica medica|
|3044106

In [42]:
sorted((
    rdd_medicos
    # Fase de map: clave especialidad, valor (sueldo,1).
    .map(lambda medico: (medico[3],(float(medico[1]),1))) 
    # Fase de Shuffle y Reduce: se agrupa por clave y se suman los valores
    .reduceByKey(lambda tupla1, tupla2: (tupla1[0] + tupla2[0],tupla1[1] + tupla2[1]))
    # Calculamos el promedio y redondeamos
    .mapValues(lambda tupla: round(tupla[0]/tupla[1],2))

).collect(),key=lambda a:-a[1])
#El resultado es luego ordenado de forma decreciente según el sueldo promedio.

[('Traumatologia', 3375880.55),
 ('Cirugia general', 3219589.92),
 ('Ginecologia', 2891897.43),
 ('Oftalmologia', 2814893.25),
 ('Otorrinolaringologia', 2641367.6),
 ('Endocrinologia', 2345923.3),
 ('Neumonologia', 2260811.35),
 ('Clinica medica', 2257325.6),
 ('Psiquiatria', 2190012.47),
 ('Pediatria', 2169721.87),
 ('Neurologia', 2077444.47),
 ('Gastroenterologia', 2069202.6),
 ('Dermatologia', 1928142.04),
 ('Cardiologia', 1906338.84)]

### Procedimiento 2: ¿Cuáles son los diagnosticos más frecuente? ¿y los menos?

In [43]:
# Cargamos los datos de turno:

df_turno = spark.read.csv("./datos/turno",header= True)

# Los convertimos a dataset distribuido:

rdd_turnos = df_turno.rdd 

In [44]:
df_turno.show()

+--------+--------------------+--------+-----------+----------+----------+-------------+-----------+------------------+
|id_turno|         diagnostico|   costo|fecha_turno|hora_turno|    estado|cuil_paciente|cuil_medico|numero_consultorio|
+--------+--------------------+--------+-----------+----------+----------+-------------+-----------+------------------+
|       1|        dolor lumbar|21791.35| 2025-03-28|  13:30:00| cancelado|  23189716769|23199216046|                 7|
|       2|  control pediatrico|23609.19| 2025-09-03|  12:30:00| realizado|  20308982724|23335880117|                33|
|       3|     dolor abdominal|28344.54| 2025-07-02|  14:00:00| realizado|  23320177575|27493041830|                31|
|       4|             cefalea|14211.18| 2026-08-14|  13:00:00|programado|  27140780130|23294623786|                 7|
|       5|          dermatitis|53269.58| 2026-11-30|  14:00:00| cancelado|  23132800297|27166306974|                16|
|       6|            embarazo|24958.26|

In [45]:
res_diagnosticos = sorted((
    rdd_turnos
    # Fase de map: clave: diagnostico, valor: 1.
    .map(lambda turno: (turno[1],1)) 
    # Fase de Shuffle y Reduce: se agrupa por clave y se suman los valores para obtener el conteo
    .reduceByKey(lambda valor1, valor2: (valor1 + valor2))
).collect(),key=lambda a:-a[1])
#El resultado es luego ordenado de forma decreciente según el conteo.

In [46]:
n = 3
print("Diagnosticos más frecuentes:\n")
for i in range(n):
    print('Se diagnosticó', res_diagnosticos[i][0], res_diagnosticos[i][1],'veces')
print("\nDiagnosticos menos frecuentes:\n")
for i in range(n):
    print('Se diagnosticó', res_diagnosticos[-i-n][0], res_diagnosticos[-i-n][1],'veces')

Diagnosticos más frecuentes:

Se diagnosticó embarazo 741 veces
Se diagnosticó dolor abdominal 724 veces
Se diagnosticó arritmia 696 veces

Diagnosticos menos frecuentes:

Se diagnosticó control pediatrico 185 veces
Se diagnosticó control cardiologico 253 veces
Se diagnosticó control ginecologico 254 veces


### Procedimiento 3: ¿Cuántos turnos tomó cada médico?

In [47]:
# usando los mismos datos que la anterior consulta:
res_cant_turnos_por_med = sorted((
    rdd_turnos
    # Fase de map: clave: cuil_medico, valor: 1.
    .map(lambda turno: (int(turno[7]),1)) 
    # Fase de Shuffle y Reduce: se agrupa por clave y se suman los valores para obtener el conteo
    .reduceByKey(lambda valor1, valor2: (valor1 + valor2))
).collect(),key=lambda a:-a[1])
#El resultado es luego ordenado de forma decreciente según el conteo.

In [48]:
print("Cantidad de turnos por médico:")
for medico in res_cant_turnos_por_med:
    print(medico)


Cantidad de turnos por médico:
(27174348300, 79)
(27220982094, 78)
(20265029699, 77)
(23199216046, 76)
(20308275900, 76)
(20120633626, 76)
(30123128347, 75)
(30249933407, 75)
(20229700690, 74)
(20304506005, 73)
(30213261742, 73)
(27480301673, 73)
(20118402854, 73)
(27286684881, 73)
(20482619303, 72)
(30287352698, 72)
(23136681318, 72)
(27418547395, 72)
(27196508168, 72)
(30430029723, 71)
(30180844395, 71)
(27146278482, 71)
(27290918689, 71)
(30294548084, 71)
(23186080620, 71)
(27430293648, 70)
(20100867374, 70)
(20240570059, 70)
(23237958748, 70)
(23153090464, 70)
(23451822881, 70)
(20218230576, 69)
(23351188756, 69)
(30455772029, 69)
(30361032815, 69)
(23294623786, 68)
(23129270187, 68)
(27130649623, 68)
(27146393286, 68)
(30434390846, 68)
(20301562664, 68)
(27169553708, 68)
(30244103636, 68)
(27493041830, 67)
(27312300622, 67)
(27393813388, 67)
(23447724150, 67)
(27171784042, 67)
(23456608821, 67)
(27380866378, 67)
(20424620896, 67)
(30345734726, 67)
(27420775470, 66)
(23139814662, 6